# `design.beams` — Simple example

Quick path through the beam module. One-shot construction with `Beam.rectangular(...)`.
Beams have no P-M diagram — they are flexural-shear elements per ACI §9 and §18.6.

In [ ]:
import matplotlib.pyplot as plt
from design import Concrete, Steel
from design.beams import Beam, BeamDemands

## 1. Build the beam in one call

All scalars in one place. `cover` is the clear cover to the outside edge of the stirrup, applied symmetrically to both top and bottom fibers.

In [ ]:
concrete = Concrete(fc=28)
steel    = Steel(fy=420, grade=60)

beam = Beam.rectangular(
    bw=300, h=600, cover=40,
    concrete=concrete, steel=steel,
    n_top=2, db_top=20,
    n_bot=4, db_bot=20,
    db_stirrup=10, n_legs=2, s_stirrup=100,
    ln=5000.0, seismic=True, Vg=80.0,
    units=6,            # kN_m_C
    label='B-1',
)
beam.summary()

## 2. Demands at three stations (i / mid / j)

Each station carries `Mu+`, `Mu-`, and `Vu`. Optionally `Vg_i`, `Vg_mid`, `Vg_j` from a gravity-only combo — if omitted, the beam's `Vg` is used at the supports.

In [ ]:
demands = BeamDemands(
    Mu_pos_i=80,    Mu_neg_i=320,   Vu_i=210,
    Mu_pos_mid=350, Mu_neg_mid=50,  Vu_mid=80,
    Mu_pos_j=80,    Mu_neg_j=320,   Vu_j=205,
)

## 3. ACI checks

In [ ]:
beam.print_checks(demands)

## 4. Three-mode design

In [ ]:
results = beam.design(demands)
beam.print_design(results, demands=demands)

In [ ]:
fig = beam.plot.dashboard(demands=demands)
plt.show()

## 5. Apply any mode and verify

`beam.apply()` accepts any of the proposals (`provided`, `minimum`, `demand`, `capacity`, `envelope`). The PROVIDED snapshot is obtained from `beam.current_proposal(demands)` so you can revert after exploring.

In [ ]:
snapshot = beam.current_proposal(demands)

beam = beam.apply(results.envelope)
chk = beam.check(demands)
print(f'After apply(envelope):  ratio M = {chk.ratio_M_overall:.3f}'
      f'   ratio V max = {chk.ratio_V_overall:.3f}'
      f'   passed = {chk.passed}')

# Revert to the original PROVIDED
beam = beam.apply(snapshot)
print(f'Reverted to PROVIDED: top {beam._infer_layer_bars("top")[1]}φ{beam._infer_layer_bars("top")[0]:.0f}')